In [43]:
!pip install pandas

In [44]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
all_records = []

start_year = datetime.now().year - 5  
end_year = datetime.now().year

for year in range(start_year, end_year + 1):

    for month in range(1, 13):

        start_date = f"{year}-{month:02d}-01"

        if month == 12:
            end_date = f"{year + 1}-01-01"
        else:
            end_date = f"{year}-{month + 1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f"JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]

            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "alert": p.get("alert"),
                "felt": p.get("felt"),
                "cdi": p.get("cdi"),
                "mmi": p.get("mmi"),
                "sig": p.get("sig"),
                "net": p.get("net"),
                "code": p.get("code"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "types": p.get("types"),
                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "type": p.get("type")
            })

df = pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.head())


Rows: 120592
Columns: 26
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  net  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   us   
1                                        Fiji region  reviewed  ...   us   
2                    103 km SW of Basco, Philippines  reviewe

In [45]:


output_file = "earthquake_data_last_5_years.csv"

df.to_csv(
    output_file,
    mode="a",
    header=not pd.io.common.file_exists(output_file),
    index=False
)

print(f"Data successfully appended to: {output_file}")
print("Rows appended:", df.shape[0])
print("Columns:", df.shape[1])

Data successfully appended to: earthquake_data_last_5_years.csv
Rows appended: 120592
Columns: 26


In [46]:


df = pd.read_csv("earthquake_data_last_5_years.csv")

print("Data successfully loaded into DataFrame")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nDataFrame:")
print(df)

print("\nFirst 5 rows:")
print(df.head())

Data successfully loaded into DataFrame
Rows: 1085068
Columns: 26

DataFrame:
                 id                     time                  updated  \
0        us6000ddi8  2021-01-31 23:20:49.923  2021-04-16 19:02:44.040   
1        us6000dev6  2021-01-31 23:08:17.161  2021-04-16 19:03:47.040   
2        us6000dev5  2021-01-31 22:54:19.760  2021-04-16 19:03:47.040   
3        us6000ddhs  2021-01-31 22:06:00.832  2021-04-16 19:02:43.040   
4        us6000dev4  2021-01-31 21:51:14.016  2021-04-16 19:03:46.040   
...             ...                      ...                      ...   
1085063  us6000typ5  2026-10-01 02:09:19.068  2026-10-01 03:56:18.040   
1085064  us6000typ0  2026-10-01 01:31:23.443  2026-10-01 01:50:10.040   
1085065  us6000tyny  2026-10-01 01:29:20.694  2026-10-01 07:41:06.409   
1085066  us6000tynp  2026-10-01 00:28:22.644  2026-10-01 00:45:24.040   
1085067  ci41341279  2026-10-01 00:01:59.630  2026-10-05 20:54:08.375   

          latitude   longitude  depth_km   ma

In [47]:



numeric_columns = df.select_dtypes(include=["number"]).columns
df[numeric_columns] = df[numeric_columns].fillna(0)


text_columns = df.select_dtypes(include=["object"]).columns
df[text_columns] = df[text_columns].fillna("Unknown")


print("Total null values after filling:", df.isnull().sum().sum())


print("\nNull values by column:")
print(df.isnull().sum())


print("\nUpdated DataFrame:")
print(df.head())

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\4144945861.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=["object"]).columns


Total null values after filling: 0

Null values by column:
id           0
time         0
updated      0
latitude     0
longitude    0
depth_km     0
mag          0
magType      0
place        0
status       0
tsunami      0
alert        0
felt         0
cdi          0
mmi          0
sig          0
net          0
code         0
ids          0
sources      0
types        0
nst          0
dmin         0
rms          0
gap          0
type         0
dtype: int64

Updated DataFrame:
           id                     time                  updated  latitude  \
0  us6000ddi8  2021-01-31 23:20:49.923  2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6  2021-01-31 23:08:17.161  2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5  2021-01-31 22:54:19.760  2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs  2021-01-31 22:06:00.832  2021-04-16 19:02:43.040   28.1524   
4  us6000dev4  2021-01-31 21:51:14.016  2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337    

In [48]:
!pip install pymysql

In [49]:
import pymysql

connection =pymysql.connect(
    host='localhost',
    user='root',
    password='Aishwaryaraghul0514',
    port = 3306,
    
)

cursor = connection.cursor()


In [50]:
cursor.execute("CREATE DATABASE IF NOT EXISTS EARTH_DATABASE")

1

In [51]:
cursor.execute("USE EARTH_DATABASE")

0

In [52]:
create_query = """
 CREATE TABLE IF NOT EXISTS earth_data(
        id VARCHAR(50) PRIMARY KEY,
        time DATETIME(3),
        updated DATETIME(3),
        latitude DECIMAL(9, 6),
        longitude DECIMAL(9, 6),
        depth_km DECIMAL(8, 2),
        mag DECIMAL(4, 2),
        magType VARCHAR(20),
        place TEXT,
        status VARCHAR(50),
        tsunami INT,
        alert VARCHAR(20),
        felt INT,
        cdi DECIMAL(4, 2),
        mmi DECIMAL(4, 2),
        sig INT,
        net VARCHAR(20),
        code VARCHAR(50),
        ids TEXT,
        sources TEXT,
        types TEXT,
        nst INT,
        dmin DECIMAL(8, 4),
        rms DECIMAL(6, 3),
        gap DECIMAL(6, 2),
        type VARCHAR(50)
    );
    """

cursor.execute(create_query)

0

In [53]:
INSERT_QUERY = """
INSERT IGNORE INTO earth_data (
    id,
    time,
    updated,
    latitude,
    longitude,
    depth_km,
    mag,
    magType,
    place,
    status,
    tsunami,
    alert,
    felt,
    cdi,
    mmi,
    sig,
    net,
    code,
    ids,
    sources,
    types,
    nst,
    dmin,
    rms,
    gap,
    type
)
VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s
)
"""

for dic in all_records:

    values = (
        dic["id"],
        dic["time"].to_pydatetime() if pd.notna(dic["time"]) else None,
        dic["updated"].to_pydatetime() if pd.notna(dic["updated"]) else None,
        dic["latitude"],
        dic["longitude"],
        dic["depth_km"],
        dic["mag"],
        dic["magType"],
        dic["place"],
        dic["status"],
        dic["tsunami"],
        dic["alert"],
        dic["felt"],
        dic["cdi"],
        dic["mmi"],
        dic["sig"],
        dic["net"],
        dic["code"],
        dic["ids"],
        dic["sources"],
        dic["types"],
        dic["nst"],
        dic["dmin"],
        dic["rms"],
        dic["gap"],
        dic["type"]
    )

    cursor.execute(INSERT_QUERY, values)

connection.commit()

print("Data insertion completed.")
print("Total records processed:", len(all_records))

Data insertion completed.
Total records processed: 120592


In [54]:
!pip install mysql query

In [55]:
import pandas as pd

In [56]:
###1. Top 10 strongest earthquakes

In [57]:
query1 = """
SELECT *
FROM earth_data
WHERE mag IS NOT NULL
ORDER BY mag DESC
LIMIT 10;
"""

df_top_10_strongest = pd.read_sql(query1, connection)

print("\n1. TOP 10 STRONGEST EARTHQUAKES")
print(df_top_10_strongest)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\2207043898.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_10_strongest = pd.read_sql(query1, connection)



1. TOP 10 STRONGEST EARTHQUAKES
             id                    time                 updated  latitude  \
0    us6000qw60 2025-07-29 23:24:52.483 2026-10-01 20:30:22.188   52.4948   
1  ak0219neiszm 2021-07-29 06:15:49.188 2026-04-15 18:42:11.216   55.3635   
2    us7000dflf 2021-03-04 19:28:33.178 2025-12-22 18:58:46.851  -29.7228   
3    us6000f53e 2021-08-12 18:35:17.231 2025-12-22 22:13:38.221  -58.3753   
4    us6000tkt2 2026-08-14 21:58:21.561 2026-09-28 16:45:50.150   -8.3358   
5    us6000jllz 2023-02-06 01:17:34.342 2026-04-13 23:32:30.032   37.2256   
6    us7000srb1 2026-06-07 23:37:41.970 2026-10-01 12:48:44.367    5.5994   
7    us7000qx2g 2025-09-18 18:58:14.939 2025-12-06 16:41:30.345   53.1426   
8    us6000dg77 2021-02-10 13:19:55.530 2025-12-22 18:54:06.521  -23.0511   
9    us6000kd0n 2023-05-19 02:57:03.172 2025-08-15 23:01:19.104  -23.2063   

   longitude  depth_km  mag magType  \
0   160.2395     35.00  8.8     mww   
1  -157.8876     35.00  8.2     mww   
2 

In [58]:
# 2. Top 10 deepest earthquakes

In [59]:
query2 = """
SELECT *
FROM earth_data
WHERE depth_km IS NOT NULL
ORDER BY depth_km DESC
LIMIT 10;
"""

df_top_10_deepest = pd.read_sql(query2, connection)

print("\n2. TOP 10 DEEPEST EARTHQUAKES")
print(df_top_10_deepest)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\2210363177.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_10_deepest = pd.read_sql(query2, connection)



2. TOP 10 DEEPEST EARTHQUAKES
           id                    time                 updated  latitude  \
0  us6000sp3p 2026-04-02 16:59:17.870 2026-06-20 17:05:22.040  -12.9316   
1  us6000k2db 2023-04-01 18:09:17.114 2023-06-03 22:06:21.040  -13.1462   
2  us7000kxdn 2023-09-18 15:35:27.270 2023-12-02 23:01:10.040  -15.1098   
3  us6000mivr 2024-03-08 22:42:23.713 2024-05-15 17:35:02.040  -19.3114   
4  us6000rk66 2025-10-29 17:07:34.157 2026-01-27 17:59:01.040  -19.0745   
5  us6000f2w3 2021-08-01 22:12:46.340 2021-10-14 01:08:25.040  -20.0989   
6  us7000q1jk 2025-05-10 02:03:16.756 2025-07-24 21:30:18.040  -17.8425   
7  us7000s0st 2026-02-14 08:27:48.418 2026-05-06 17:53:52.040  -19.0361   
8  us6000ta9m 2026-07-05 14:22:25.070 2026-09-19 21:50:28.040  -20.0123   
9  us6000tknp 2026-08-10 16:50:09.877 2026-08-23 20:59:29.040  -12.9179   

   longitude  depth_km  mag magType                        place    status  \
0   169.1934    683.58  4.0      mb  206 km ENE of Sola, Vanuatu 

In [60]:
# 3. Shallow earthquakes < 50 km and magnitude > 7.5

In [61]:
query3 = """
SELECT *
FROM earth_data
WHERE depth_km < 50
AND mag > 7.5
ORDER BY mag DESC;
"""

df_shallow_strong = pd.read_sql(query3, connection)

print("\n3. SHALLOW EARTHQUAKES < 50 KM AND MAG > 7.5")
print(df_shallow_strong)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\3389723896.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_shallow_strong = pd.read_sql(query3, connection)



3. SHALLOW EARTHQUAKES < 50 KM AND MAG > 7.5
              id                    time                 updated  latitude  \
0     us6000qw60 2025-07-29 23:24:52.483 2026-10-01 20:30:22.188   52.4948   
1   ak0219neiszm 2021-07-29 06:15:49.188 2026-04-15 18:42:11.216   55.3635   
2     us6000f53e 2021-08-12 18:35:17.231 2025-12-22 22:13:38.221  -58.3753   
3     us7000dflf 2021-03-04 19:28:33.178 2025-12-22 18:58:46.851  -29.7228   
4     us6000jllz 2023-02-06 01:17:34.342 2026-04-13 23:32:30.032   37.2256   
5     us6000tkt2 2026-08-14 21:58:21.561 2026-09-28 16:45:50.150   -8.3358   
6     us7000qx2g 2025-09-18 18:58:14.939 2025-12-06 16:41:30.345   53.1426   
7     us6000dg77 2021-02-10 13:19:55.530 2025-12-22 18:54:06.521  -23.0511   
8     us6000kd0n 2023-05-19 02:57:03.172 2025-08-15 23:01:19.104  -23.2063   
9     us7000pn9s 2025-03-28 06:20:52.715 2026-07-17 19:33:31.066   22.0110   
10    us6000rgf4 2025-10-10 20:29:20.075 2026-01-01 23:11:08.040  -60.1875   
11    us6000rtdt 2

In [62]:
# 4. Average magnitude per magnitude type

In [63]:
query4 = """
SELECT
    magType,
    AVG(mag) AS average_magnitude,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE mag IS NOT NULL
AND magType IS NOT NULL
GROUP BY magType
ORDER BY average_magnitude DESC;
"""

df_avg_mag_type = pd.read_sql(query4, connection)

print("\n4. AVERAGE MAGNITUDE PER MAGNITUDE TYPE")
print(df_avg_mag_type)


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\3068297952.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_avg_mag_type = pd.read_sql(query4, connection)



4. AVERAGE MAGNITUDE PER MAGNITUDE TYPE
       magType  average_magnitude  earthquake_count
0          mwc           6.150000                 2
1        ms_20           5.800000                 1
2          mwb           5.790625                32
3          mww           5.364216              7443
4          mwp           5.250000                 1
5        ms_vx           4.650000                 4
6           mb           4.423636             85476
7          mwr           4.323626              2827
8           mh           4.100000                 1
9           mw           3.935804               684
10         mlr           3.480000                10
11         mlv           3.468000                 5
12          md           3.437614              6567
13  ml(texnet)           3.367442                43
14         mlg           3.300000                 1
15          ml           3.275079             17373
16       mb_lg           3.200000               127


In [64]:
# 5. Year with most earthquakes

In [65]:
query5 = """
SELECT
    YEAR(time) AS earthquake_year,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE time IS NOT NULL
GROUP BY YEAR(time)
ORDER BY earthquake_count DESC
LIMIT 1;
"""

df_year_most_earthquakes = pd.read_sql(query5, connection)

print("\n5. YEAR WITH MOST EARTHQUAKES")
print(df_year_most_earthquakes)


5. YEAR WITH MOST EARTHQUAKES
   earthquake_year  earthquake_count
0             2025             22825


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\260855972.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_year_most_earthquakes = pd.read_sql(query5, connection)


In [66]:
# 6. Month with highest number of earthquakes

In [67]:
query6 = """
SELECT
    MONTH(time) AS month_number,
    MONTHNAME(time) AS month_name,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE time IS NOT NULL
GROUP BY MONTH(time), MONTHNAME(time)
ORDER BY earthquake_count DESC
LIMIT 1;
"""

df_month_most_earthquakes = pd.read_sql(query6, connection)

print("\n6. MONTH WITH MOST EARTHQUAKES")
print(df_month_most_earthquakes)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\3792669781.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_month_most_earthquakes = pd.read_sql(query6, connection)



6. MONTH WITH MOST EARTHQUAKES
   month_number month_name  earthquake_count
0             7       July             11847


In [68]:
# 7. Day of week with most earthquakes

In [69]:
query7 = """
SELECT
    DAYOFWEEK(time) AS day_number,
    DAYNAME(time) AS day_name,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE time IS NOT NULL
GROUP BY DAYOFWEEK(time), DAYNAME(time)
ORDER BY earthquake_count DESC
LIMIT 1;
"""

df_day_most_earthquakes = pd.read_sql(query7, connection)

print("\n7. DAY OF WEEK WITH MOST EARTHQUAKES")
print(df_day_most_earthquakes)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\2991939977.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_day_most_earthquakes = pd.read_sql(query7, connection)



7. DAY OF WEEK WITH MOST EARTHQUAKES
   day_number day_name  earthquake_count
0           6   Friday             17442


In [70]:
# 8. Count of earthquakes per hour

In [71]:
query8 = """
SELECT
    HOUR(time) AS hour_of_day,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE time IS NOT NULL
GROUP BY HOUR(time)
ORDER BY hour_of_day;
"""

df_hourly_earthquakes = pd.read_sql(query8, connection)

print("\n8. EARTHQUAKES PER HOUR")
print(df_hourly_earthquakes)


8. EARTHQUAKES PER HOUR
    hour_of_day  earthquake_count
0             0              4975
1             1              5284
2             2              5125
3             3              5312
4             4              5276
5             5              4960
6             6              4875
7             7              5029
8             8              5036
9             9              4958
10           10              4879
11           11              4744
12           12              4725
13           13              4970
14           14              4822
15           15              4946
16           16              4872
17           17              4990
18           18              5033
19           19              5085
20           20              5135
21           21              5340
22           22              5235
23           23              4991


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\4104324140.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hourly_earthquakes = pd.read_sql(query8, connection)


In [72]:
# 9. Most active reporting network

In [73]:
query9 = """
SELECT
    net,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE net IS NOT NULL
GROUP BY net
ORDER BY earthquake_count DESC
LIMIT 1;
"""

df_active_network = pd.read_sql(query9, connection)

print("\n9. MOST ACTIVE REPORTING NETWORK")
print(df_active_network)


9. MOST ACTIVE REPORTING NETWORK
  net  earthquake_count
0  us            105470


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\1885005954.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_active_network = pd.read_sql(query9, connection)


In [74]:
# 10. All reporting networks ranked by activity

In [75]:
query10 = """
SELECT
    net,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE net IS NOT NULL
GROUP BY net
ORDER BY earthquake_count DESC;
"""

df_network_activity = pd.read_sql(query10, connection)

print("\n10. REPORTING NETWORK ACTIVITY")
print(df_network_activity)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\2544650029.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_network_activity = pd.read_sql(query10, connection)



10. REPORTING NETWORK ACTIVITY
       net  earthquake_count
0       us            105470
1       pr              6554
2       ak              3875
3       tx              1040
4       nc              1002
5       hv               943
6       ci               914
7       nn               464
8       uu               116
9       ok                93
10      uw                60
11      nm                22
12      se                18
13  iscgem                16
14      av                 6
15      ew                 2
16      mb                 2


In [76]:
# 11. Top 5 places with highest casualties

In [77]:
query_11 = """
SELECT 
    place,
    SUM(casualties) AS total_casualties
FROM earth_data
WHERE casualties IS NOT NULL
GROUP BY place
ORDER BY total_casualties DESC
LIMIT 5;
"""

result_11 = pd.read_sql(query_11, connection)

print("\n11. Top 5 Places with Highest Casualties")
print(result_11)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_5908\3390525047.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_11 = pd.read_sql(query_11, connection)


OperationalError: (1054, "Unknown column 'casualties' in 'field list'")

In [ ]:
import pandas as pd

In [ ]:
# 12. Total estimated economic loss per continent

In [ ]:
# 13. Average economic loss by alert level

In [ ]:
# 14. Reviewed vs automatic earthquakes

In [ ]:
query14 = """
SELECT
    status,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE status IS NOT NULL
GROUP BY status
ORDER BY earthquake_count DESC;
"""

df_status = pd.read_sql(query14, connection)

print("\n14. REVIEWED VS AUTOMATIC EARTHQUAKES")
print(df_status)


14. REVIEWED VS AUTOMATIC EARTHQUAKES
      status  earthquake_count
0   reviewed            120373
1  automatic               176


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\1846867001.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_status = pd.read_sql(query14, connection)


In [ ]:
# 15. Count by earthquake type

In [ ]:
query15 = """
SELECT
    type,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE type IS NOT NULL
GROUP BY type
ORDER BY earthquake_count DESC;
"""

df_type = pd.read_sql(query15, connection)

print("\n15. EARTHQUAKES BY TYPE")
print(df_type)


15. EARTHQUAKES BY TYPE
                     type  earthquake_count
0              earthquake            119647
1        mining explosion               855
2               ice quake                13
3       volcanic eruption                12
4               Landslide                 6
5             other event                 5
6  experimental explosion                 5
7           mine collapse                 3
8            quarry blast                 2
9               explosion                 1


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\4142576283.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_type = pd.read_sql(query15, connection)


In [ ]:
# 16. Number of earthquakes by data type

In [ ]:
query16 = """
SELECT
    types,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE types IS NOT NULL
GROUP BY types
ORDER BY earthquake_count DESC;
"""

df_data_types = pd.read_sql(query16, connection)

print("\n16. EARTHQUAKES BY DATA TYPE")
print(df_data_types)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\2141251623.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_data_types = pd.read_sql(query16, connection)



16. EARTHQUAKES BY DATA TYPE
                                                 types  earthquake_count
0                                  ,origin,phase-data,             91689
1                             ,dyfi,origin,phase-data,              9115
2                         ,origin,phase-data,shakemap,              2652
3                  ,earthquake-name,origin,phase-data,              2172
4                    ,moment-tensor,origin,phase-data,              1435
..                                                 ...               ...
581  ,associate,dyfi,losspager,moment-tensor,origin...                 1
582  ,dyfi,earthquake-name,impact-link,losspager,mo...                 1
583  ,associate,dyfi,earthquake-name,event-sequence...                 1
584  ,dyfi,moment-tensor,origin,phase-data,shake-al...                 1
585  ,dyfi,event-sequence,losspager,moment-tensor,o...                 1

[586 rows x 2 columns]


In [ ]:
# 17. Average RMS and gap per continent

In [ ]:
# 18. Events with high station coverage

In [ ]:
query18 = """
SELECT
    id,
    time,
    place,
    mag,
    nst
FROM earth_data
WHERE nst > 100
ORDER BY nst DESC;
"""

df_high_station = pd.read_sql(query18, connection)

print("\n18. EVENTS WITH HIGH STATION COVERAGE")
print(df_high_station)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\2518993554.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_high_station = pd.read_sql(query18, connection)



18. EVENTS WITH HIGH STATION COVERAGE
              id                    time  \
0     us6000m12f 2024-01-02 01:17:31.568   
1     us6000qzfl 2025-08-09 08:01:41.017   
2     us7000rluk 2026-01-01 06:46:54.742   
3     us7000pvtr 2025-04-29 14:53:37.897   
4     usd001097k 2023-12-11 18:35:59.877   
...          ...                     ...   
8052  us6000qubl 2025-07-23 08:25:29.283   
8053  us6000qubz 2025-07-23 10:02:21.981   
8054  us6000qvk1 2025-07-28 03:54:33.561   
8055  us6000qwns 2025-07-30 18:34:24.189   
8056  us6000qwss 2025-07-31 01:28:00.548   

                                                  place  mag  nst  
0                             11 km W of Anamizu, Japan  5.4  619  
1                     120 km ENE of Ozernovskiy, Russia  5.0  566  
2                           111 km N of Yakutat, Alaska  5.7  516  
3                               Macquarie Island region  6.8  475  
4     49 km WNW of San Antonio de los Cobres, Argentina  5.5  466  
...                     

In [ ]:
# 19. Number of tsunamis triggered per year

In [ ]:
query19 = """
SELECT
    YEAR(time) AS earthquake_year,
    COUNT(*) AS tsunami_count
FROM earth_data
WHERE tsunami = 1
AND time IS NOT NULL
GROUP BY YEAR(time)
ORDER BY earthquake_year;
"""

df_tsunami_year = pd.read_sql(query19, connection)

print("\n19. TSUNAMIS TRIGGERED PER YEAR")
print(df_tsunami_year)


19. TSUNAMIS TRIGGERED PER YEAR
   earthquake_year  tsunami_count
0             2021            114
1             2022            136
2             2023            119
3             2024            114
4             2025            142
5             2026             45


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\4094439808.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tsunami_year = pd.read_sql(query19, connection)


In [ ]:
# 20. Count earthquakes by alert level

In [ ]:
query20 = """
SELECT
    alert,
    COUNT(*) AS earthquake_count
FROM earth_data
WHERE alert IS NOT NULL
GROUP BY alert
ORDER BY earthquake_count DESC;
"""

df_alerts = pd.read_sql(query20, connection)

print("\n20. EARTHQUAKES BY ALERT LEVEL")
print(df_alerts)


20. EARTHQUAKES BY ALERT LEVEL
     alert  earthquake_count
0  Unknown            115617
1    green              4721
2   yellow               140
3      red                28
4   orange                27


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\3663855485.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_alerts = pd.read_sql(query20, connection)


In [ ]:
# 21. Top 5 countries by average magnitude

In [ ]:
query_21 = """
SELECT
    place,
    AVG(mag) AS average_magnitude
FROM earth_data
WHERE place IS NOT NULL
  AND mag IS NOT NULL
  AND time >= DATE_SUB(CURDATE(), INTERVAL 5 YEAR)
GROUP BY place
ORDER BY average_magnitude DESC
LIMIT 5;
"""

result_21 = pd.read_sql(query_21, connection)

print("\n21. Top 5 Places with Highest Average Magnitude in the Past 5 Years")
print(result_21)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\2519981213.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_21 = pd.read_sql(query_21, connection)



21. Top 5 Places with Highest Average Magnitude in the Past 5 Years
                                               place  average_magnitude
0        2025 Kamchatka Peninsula, Russia Earthquake                8.8
1                  25 km SW of Kablalan, Philippines                7.8
2  Pazarcik earthquake, Kahramanmaras earthquake ...                7.8
3          2025 Mandalay, Burma (Myanmar) Earthquake                7.7
4          210 km SSW of George Town, Cayman Islands                7.6


In [ ]:
# 22. Countries with both shallow and deep earthquakes in the same month

In [ ]:
query_22 = """
SELECT
    place,
    YEAR(time) AS year,
    MONTH(time) AS month,
    MIN(depth_km) AS minimum_depth,
    MAX(depth_km) AS maximum_depth
FROM earth_data
WHERE place IS NOT NULL
  AND depth_km IS NOT NULL
GROUP BY place, YEAR(time), MONTH(time)
HAVING MIN(depth_km) < 70
   AND MAX(depth_km) > 300
ORDER BY place, year, month;
"""

result_22 = pd.read_sql(query_22, connection)

print("\n22. Places That Experienced Both Shallow and Deep Earthquakes in the Same Month")
print(result_22)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\1061132014.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_22 = pd.read_sql(query_22, connection)



22. Places That Experienced Both Shallow and Deep Earthquakes in the Same Month
                             place  year  month  minimum_depth  maximum_depth
0                        Banda Sea  2021      8          54.22         589.76
1                        Banda Sea  2022      2          10.00         472.22
2                        Banda Sea  2022     12           8.74         509.86
3                        Banda Sea  2023      2          21.58         334.94
4                        Banda Sea  2023      4           4.00         416.25
..                             ...   ...    ...            ...            ...
354                 Vanuatu region  2026      8          10.00         629.75
355                 Vanuatu region  2026      9          23.97         665.06
356  Volcano Islands, Japan region  2024     11           8.00         327.49
357  Volcano Islands, Japan region  2025      1          10.00         534.48
358  Volcano Islands, Japan region  2025     12          40.3

In [ ]:
# 23. Year-over-year growth rate

In [ ]:
query23 = """
WITH yearly_counts AS (
    SELECT
        YEAR(time) AS earthquake_year,
        COUNT(*) AS earthquake_count
    FROM earth_data
    WHERE time IS NOT NULL
    GROUP BY YEAR(time)
)

SELECT
    earthquake_year,
    earthquake_count,
    LAG(earthquake_count) OVER (
        ORDER BY earthquake_year
    ) AS previous_year_count,

    ROUND(
        (
            (earthquake_count -
             LAG(earthquake_count) OVER (ORDER BY earthquake_year))
            /
            NULLIF(
                LAG(earthquake_count) OVER (ORDER BY earthquake_year),
                0
            )
        ) * 100,
        2
    ) AS yoy_growth_percent

FROM yearly_counts
ORDER BY earthquake_year;
"""

df_yoy_growth = pd.read_sql(query23, connection)

print("\n23. YEAR-OVER-YEAR EARTHQUAKE GROWTH")
print(df_yoy_growth)


23. YEAR-OVER-YEAR EARTHQUAKE GROWTH
   earthquake_year  earthquake_count  previous_year_count  yoy_growth_percent
0             2021             21926                  NaN                 NaN
1             2022             20208              21926.0               -7.84
2             2023             20830              20208.0                3.08
3             2024             18659              20830.0              -10.42
4             2025             22825              18659.0               22.33
5             2026             16101              22825.0              -29.46


C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\305850818.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_yoy_growth = pd.read_sql(query23, connection)


In [ ]:
# 24. Three most seismically active regions

In [ ]:
query_24 = """
SELECT 
    place,
    COUNT(*) AS earthquake_frequency,
    AVG(mag) AS average_magnitude,
    ROUND(COUNT(*) * AVG(mag), 2) AS seismic_activity_score
FROM earth_data
WHERE place IS NOT NULL
  AND mag IS NOT NULL
GROUP BY place
ORDER BY seismic_activity_score DESC
LIMIT 3;
"""

result_24 = pd.read_sql(query_24, connection)

print("\n24. Top 3 Seismically Active Regions")
print(result_24)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\3260806615.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_24 = pd.read_sql(query_24, connection)



24. Top 3 Seismically Active Regions
                           place  earthquake_frequency  average_magnitude  \
0  South Sandwich Islands region                  3223           4.682116   
1        Kermadec Islands region                  1997           4.645904   
2      south of the Fiji Islands                  2080           4.444423   

   seismic_activity_score  
0                15090.46  
1                 9277.87  
2                 9244.40  


In [ ]:
# 25. Average depth within ±5° latitude of equator

In [ ]:
query_25 = """
SELECT
    place,
    AVG(depth_km) AS average_depth
FROM earth_data
WHERE place IS NOT NULL
  AND latitude BETWEEN -5 AND 5
  AND depth_km IS NOT NULL
GROUP BY place
ORDER BY average_depth DESC;
"""

result_25 = pd.read_sql(query_25, connection)

print("\n25. Average Depth of Earthquakes Within ±5° of the Equator")
print(result_25)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\1982975804.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_25 = pd.read_sql(query_25, connection)



25. Average Depth of Earthquakes Within ±5° of the Equator
                                     place  average_depth
0            228 km NNE of Palu, Indonesia         646.54
1     86 km NNW of Kimbe, Papua New Guinea         613.79
2     89 km NNW of Kimbe, Papua New Guinea         608.94
3      234 km SSE of Tabiauan, Philippines         604.81
4     88 km NNW of Kimbe, Papua New Guinea         604.53
...                                    ...            ...
6687       22 km NNE of Sibolga, Indonesia           3.60
6688         193 km NNE of Palu, Indonesia           3.44
6689         44 km NNW of Salinas, Ecuador           3.06
6690        9 km N of San Gabriel, Ecuador           2.54
6691       37 km NNW of Archidona, Ecuador           1.70

[6692 rows x 2 columns]


In [ ]:
# 26. HIGHEST SHALLOW-TO-DEEP EARTHQUAKE RATIO

In [ ]:
query_26 = """
SELECT 
    place,

    SUM(
        CASE 
            WHEN depth_km < 70 THEN 1
            ELSE 0
        END
    ) AS shallow_count,

    SUM(
        CASE 
            WHEN depth_km > 300 THEN 1
            ELSE 0
        END
    ) AS deep_count,

    ROUND(
        SUM(
            CASE 
                WHEN depth_km < 70 THEN 1
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE 
                    WHEN depth_km > 300 THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS shallow_deep_ratio

FROM earth_data

WHERE place IS NOT NULL

GROUP BY place

HAVING deep_count > 0

ORDER BY shallow_deep_ratio DESC

LIMIT 10;
"""

result_26 = pd.read_sql(query_26, connection)

print("\n26. Places with Highest Shallow-to-Deep Earthquake Ratio")
print(result_26)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\2541139487.py:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_26 = pd.read_sql(query_26, connection)



26. Places with Highest Shallow-to-Deep Earthquake Ratio
                                           place  shallow_count  deep_count  \
0                  Kermadec Islands, New Zealand          805.0         5.0   
1                  Volcano Islands, Japan region          293.0         3.0   
2                  south of the Kermadec Islands          690.0        17.0   
3           New Britain region, Papua New Guinea           32.0         1.0   
4  Maug Islands region, Northern Mariana Islands          303.0        20.0   
5                                    Guam region           13.0         1.0   
6                                          Tonga           25.0         2.0   
7           New Ireland region, Papua New Guinea           11.0         1.0   
8                        Kermadec Islands region         1673.0       176.0   
9                          Mindanao, Philippines           27.0         3.0   

   shallow_deep_ratio  
0              161.00  
1               97.67  


In [ ]:
# 27. MAGNITUDE DIFFERENCE: TSUNAMI VS NON-TSUNAMI

In [ ]:
query_27 = """
SELECT

    AVG(
        CASE
            WHEN tsunami = 1 THEN mag
        END
    ) AS avg_tsunami_magnitude,

    AVG(
        CASE
            WHEN tsunami = 0 THEN mag
        END
    ) AS avg_non_tsunami_magnitude,

    AVG(
        CASE
            WHEN tsunami = 1 THEN mag
        END
    )
    -
    AVG(
        CASE
            WHEN tsunami = 0 THEN mag
        END
    ) AS magnitude_difference

FROM earth_data

WHERE mag IS NOT NULL;
"""

result_27 = pd.read_sql(query_27, connection)

print("\n27. Average Magnitude Difference")
print(result_27)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\725912210.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_27 = pd.read_sql(query_27, connection)



27. Average Magnitude Difference
   avg_tsunami_magnitude  avg_non_tsunami_magnitude  magnitude_difference
0               5.414597                     4.2496              1.164997


In [ ]:
# 28. Events with lowest data reliability

In [ ]:
query28 = """
SELECT
    id,
    time,
    place,
    mag,
    rms,
    gap,
    ROUND((rms + gap), 2) AS error_score
FROM earth_data
WHERE rms IS NOT NULL
AND gap IS NOT NULL
ORDER BY error_score DESC
LIMIT 10;
"""

df_low_reliability = pd.read_sql(query28, connection)

print("\n28. LOWEST DATA RELIABILITY EVENTS")
print(df_low_reliability)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_2480\117009649.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_low_reliability = pd.read_sql(query28, connection)



28. LOWEST DATA RELIABILITY EVENTS
              id                    time  \
0     pr71519528 2026-06-11 12:21:59.730   
1     nn00897599 2025-05-14 12:02:09.568   
2   pr2024051000 2024-02-20 05:23:09.140   
3     pr71508583 2026-02-23 19:35:44.000   
4     us7000denb 2021-02-14 19:26:12.202   
5  aka2026fwzivr 2026-03-25 08:04:49.491   
6   pr2022227000 2022-08-15 01:27:45.500   
7     pr71489653 2025-07-18 23:19:06.230   
8   pr2022289000 2022-10-16 20:43:51.670   
9   pr2021003002 2021-01-03 01:55:58.430   

                                               place   mag   rms     gap  \
0         95 km ENE of Cruz Bay, U.S. Virgin Islands  3.31  0.14  359.00   
1                          60 km NE of Valmy, Nevada  3.00  0.16  358.18   
2         77 km NW of Sandy Ground Village, Anguilla  3.58  0.16  358.00   
3        86 km WNW of Sandy Ground Village, Anguilla  3.17  0.22  356.00   
4        Andreanof Islands, Aleutian Islands, Alaska  3.30  0.34  355.00   
5                      

In [ ]:
# 29. Consecutive earthquakes within 50 km and 1 hour

In [ ]:
# 30. Regions with highest frequency of deep-focus earthquakes

In [ ]:
query_30 = """
SELECT
    place,
    COUNT(*) AS deep_earthquake_count,
    AVG(depth_km) AS average_depth

FROM earth_data

WHERE depth_km > 300
  AND place IS NOT NULL

GROUP BY place

ORDER BY deep_earthquake_count DESC

LIMIT 10;
"""

result_30 = pd.read_sql(query_30, connection)

print("\n30. Regions with Highest Frequency of Deep-Focus Earthquakes")
print(result_30)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\618360290.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_30 = pd.read_sql(query_30, connection)



30. Regions with Highest Frequency of Deep-Focus Earthquakes
                                    place  deep_earthquake_count  \
0               south of the Fiji Islands                   1747   
1                             Fiji region                   1311   
2                 Kermadec Islands region                    176   
3             Bonin Islands, Japan region                    140   
4               Izu Islands, Japan region                    111   
5  Pagan region, Northern Mariana Islands                     69   
6                          Vanuatu region                     57   
7                  Mariana Islands region                     45   
8          northwest of the Kuril Islands                     31   
9                               Banda Sea                     20   

   average_depth  
0     522.863635  
1     556.938795  
2     393.957955  
3     465.806786  
4     427.046036  
5     453.486087  
6     610.027368  
7     314.243333  
8     412.031935  

In [ ]:
pip install streamlit pandas mysql-connector-python plotly

   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.3 MB 600.1 kB/s eta 0:00:17
   -- ------------------------------------- 0.5/10.3 MB 600.1 kB/s eta 0:00:17
   --- ------------------------------------ 0.8/10.3 MB 624.5 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/10.3 MB 624.5 kB/s eta 0:00:16
   ---- ----------------------------------- 1.0/10.3 MB 654.1 kB/s eta 0:00:15
   ---- ----------------------------------- 1.0/10.3 MB 654.1 kB/s eta 0:00:15
   ----- ---------------------------------- 1.3/10.3 MB 673.7 kB/s eta 0:00:14
   ------ --------------------------------- 1.6/10.3 MB 709.5 kB/s eta 0:00:13
   ------- -------

In [ ]:
columns = pd.read_sql("DESCRIBE earth_data", connection)

st.subheader("Database Columns")
st.dataframe(columns)

C:\Users\JANAASHA\AppData\Local\Temp\ipykernel_11716\3658265179.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  columns = pd.read_sql("DESCRIBE earth_data", connection)
2026-10-06 08:42:19.699 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 08:42:19.702 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 08:42:19.706 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 08:42:19.718 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 08:42:19.721 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-06 08:42:19.723 Thread 'MainThre

DeltaGenerator()

In [ ]:
%%writefile app.py
st.title("🌎Earthquake Data Analysis Dashboard")

st.write(
    "Select any problem statement (1-30) to run the corresponding SQL query."
)

# Dropdown
task = st.selectbox(
    "Choose Task Number",
    list(queries.keys())
)

# Run button
if st.button("Run Query"):

    query = queries[task]

    df = pd.read_sql(
        query,
        engine
    )

    st.subheader(
        f"Results for: {task}"
    )

    st.dataframe(
        df,
        use_container_width=True
    )


Overwriting app.py


!streamlit run app.py